# Customer Support Ticket Classifier — Evaluated AI Prototype

**Use case:** Classify customer-support messages into `Billing`, `Technical Issue`, `Account Access`, or `General Inquiry`.

This notebook demonstrates a pretrained zero-shot model, a small labelled evaluation set, failure-case analysis, and an optional Gradio interface.

> **Important:** The included examples are a tiny, hand-written demonstration set, not a representative benchmark. Report the metrics produced by your run; do not claim that illustrative/example scores prove real-world performance.

## 1. Install dependencies

Run this cell in Google Colab. The first model load downloads weights and may take a few minutes.

In [ ]:
%pip -q install transformers torch scikit-learn pandas matplotlib seaborn gradio

## 2. Load the model and define labels

The model uses **zero-shot classification**: it scores the candidate labels for a message without training on this notebook's examples. Scores are model scores, not calibrated probabilities.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

LABELS = [
    "Billing",
    "Technical Issue",
    "Account Access",
    "General Inquiry",
]

MODEL_NAME = "facebook/bart-large-mnli"

classifier = pipeline(
    task="zero-shot-classification",
    model=MODEL_NAME,
)

def predict_ticket(message, candidate_labels=LABELS):
    message = str(message).strip()
    if not message:
        return {
            "message": message,
            "category": "Needs human review",
            "score": 0.0,
            "scores": {},
            "review": True,
            "reason": "Empty message",
        }

    result = classifier(
        sequences=message,
        candidate_labels=candidate_labels,
        multi_label=False,
    )
    scores = dict(zip(result["labels"], result["scores"]))
    category = result["labels"][0]
    score = float(result["scores"][0])

    # Heuristic review gate, not a statistically calibrated confidence threshold.
    review = score < 0.60
    return {
        "message": message,
        "category": category,
        "score": score,
        "scores": scores,
        "review": review,
        "reason": "Top score below 0.60" if review else "No automatic review flag",
    }

## 3. Build a labelled evaluation set

The labels below are expected answers written for demonstration. Some examples are intentionally ambiguous or out of scope. Treat them as qualitative test cases, not as a statistically representative dataset.

In [ ]:
evaluation_data = [
    ("I was charged twice for my monthly subscription.", "Billing", "clear"),
    ("Please explain the extra fee on my invoice.", "Billing", "clear"),
    ("My refund has not arrived yet.", "Billing", "clear"),
    ("I need to update my payment method.", "Billing", "clear"),
    ("The application crashes when I upload a PDF.", "Technical Issue", "clear"),
    ("The dashboard shows a server error.", "Technical Issue", "clear"),
    ("The page freezes whenever I click Save.", "Technical Issue", "clear"),
    ("I cannot download my report because the app fails.", "Technical Issue", "clear"),
    ("I forgot my password and cannot sign in.", "Account Access", "clear"),
    ("My account is locked after several login attempts.", "Account Access", "clear"),
    ("The password reset link has expired.", "Account Access", "clear"),
    ("I lost access to the email used for login.", "Account Access", "clear"),
    ("What time does customer support open?", "General Inquiry", "clear"),
    ("Where can I read the user guide?", "General Inquiry", "clear"),
    ("What features are included in the service?", "General Inquiry", "clear"),
    ("How can I contact your team?", "General Inquiry", "clear"),
    ("I was charged twice and now the app crashes.", "Billing", "mixed"),
    ("I cannot log in and need to change my subscription.", "Account Access", "mixed"),
    ("Help! It is not working.", "Technical Issue", "vague"),
    ("I need help with my account.", "Account Access", "vague"),
    ("Tell me a joke about penguins.", "General Inquiry", "out-of-scope"),
    ("My invoice is wrong and the login page is broken.", "Billing", "mixed"),
]

eval_df = pd.DataFrame(
    evaluation_data,
    columns=["message", "expected", "case_type"]
)
eval_df

## 4. Run evaluation

Review overall accuracy and macro F1, then inspect per-class precision and recall. With so few examples, metrics can change substantially when one example changes.

In [ ]:
predictions = [predict_ticket(message) for message in eval_df["message"]]
eval_df["predicted"] = [p["category"] for p in predictions]
eval_df["top_score"] = [p["score"] for p in predictions]
eval_df["flag_for_review"] = [p["review"] for p in predictions]

print(f"Accuracy: {accuracy_score(eval_df['expected'], eval_df['predicted']):.3f}")
print("\nClassification report:")
print(classification_report(
    eval_df["expected"],
    eval_df["predicted"],
    labels=LABELS,
    zero_division=0,
))

cm = confusion_matrix(eval_df["expected"], eval_df["predicted"], labels=LABELS)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("Predicted label")
plt.ylabel("Expected label")
plt.title("Confusion matrix — small demonstration set")
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

eval_df

## 5. Failure-case analysis

These cases test ambiguous, mixed-intent, vague, and out-of-scope messages. A single-label classifier must choose one category even when a human might ask a follow-up question or assign multiple tags. Inspect the outputs rather than assuming the model is correct.

In [ ]:
failure_examples = [
    "It is broken.",
    "My account is locked and I was billed twice.",
    "Can you help me?",
    "Tell me a joke about penguins.",
    "I cannot log in because the website crashes after I enter my password.",
]

failure_rows = []
for message in failure_examples:
    p = predict_ticket(message)
    failure_rows.append({
        "input": message,
        "predicted_category": p["category"],
        "top_score": round(p["score"], 3),
        "review_flag": p["review"],
        "review_reason": p["reason"],
        "all_label_scores": {k: round(v, 3) for k, v in p["scores"].items()},
    })

pd.DataFrame(failure_rows)

### What to do when the model fails

- **Vague message:** ask the user for more detail instead of routing automatically.
- **Mixed intent:** allow multiple labels or ask which issue is most urgent.
- **Out-of-scope request:** add an explicit `Other / Out of scope` label if the application requires it.
- **Low top score:** flag for human review. The 0.60 cutoff here is only a demonstration heuristic; tune it on a separate validation set.
- **Sensitive information:** avoid collecting passwords, payment-card details, or unnecessary personal data.

The model's score is not a calibrated probability. A high score does not guarantee correctness.

## 6. Prompt/model evaluation examples

This is a zero-shot model rather than a chat API, so there is no generative prompt to tune. The candidate labels and message wording act as the task specification. Compare the same test set across model versions or alternative label descriptions and record the resulting metrics.

Suggested experiment log:

| Experiment | Change | What to compare |
|---|---|---|
| A | Original four labels | Accuracy, macro F1, per-class recall |
| B | Add `Other / Out of scope` | Whether unrelated requests are better handled |
| C | Use label descriptions, e.g. `Billing: invoices, charges, refunds` | Whether category confusions decrease |
| D | Change review threshold on validation data | Human-review rate versus missed errors |

Do not select a threshold using the test set and then report the same test set as an unbiased final evaluation.

## 7. Interactive Gradio interface

Run the cell below to launch a simple interface. In Colab, Gradio normally returns a temporary public URL when `share=True`; use it only for demo data and do not enter private customer information.

In [ ]:
import gradio as gr

def gradio_classify(message):
    if not str(message).strip():
        return "Please enter a message.", {}, "Human review: yes (empty input)"

    p = predict_ticket(message)
    status = (
        "Human review recommended"
        if p["review"]
        else "No review flag (still verify important cases)"
    )
    return (
        p["category"],
        {k: float(v) for k, v in p["scores"].items()},
        f"Top score: {p['score']:.1%}\n{status}\nReason: {p['reason']}",
    )

demo = gr.Interface(
    fn=gradio_classify,
    inputs=gr.Textbox(
        lines=3,
        label="Customer support message",
        placeholder="Example: I was charged twice this month.",
    ),
    outputs=[
        gr.Textbox(label="Predicted category"),
        gr.Label(label="Scores by category"),
        gr.Textbox(label="Review recommendation"),
    ],
    title="Customer Support Ticket Classifier",
    description=(
        "Classifies messages into Billing, Technical Issue, Account Access, "
        "or General Inquiry. Predictions are assistive and may be wrong."
    ),
    examples=[
        ["I was charged twice this month."],
        ["The application crashes when I upload a file."],
        ["I forgot my password and cannot log in."],
        ["What are your customer-support hours?"],
        ["It is broken."],
    ],
)

demo.launch(share=True, debug=False)

## 8. Example input/output (illustrative)

**Input:** `I forgot my password and cannot log in.`

**Expected category:** `Account Access`

The actual predicted category and scores are generated by the model when you run the notebook. Do not hard-code sample confidence values into the report.

## 9. Success criteria

A practical first-pass target is accuracy ≥ 0.85 and macro F1 ≥ 0.80 on a larger, held-out, representative labelled dataset. This tiny demonstration set is for workflow verification only and is too small to establish that the targets have been met reliably.

**Prototype limitation:** zero-shot scores can be miscalibrated, category descriptions affect predictions, and mixed-intent messages may need multi-label classification or human triage.